# RC9.2.2 RC6 — production schedule on Colab
## Variant B: with Google Drive (recommended)

Release `L6.3.2.8-RC9.2.2-PRODUCTION-HARDENED-RC6`. Full instructions:
`PRODUCTION_RUN_GUIDE.md` in the package root.

**What this notebook does**
1. Installs the pinned solver (OR-Tools 9.15.6755), scipy and ruff.
2. Mounts Drive and finds the package ZIP there.
3. Runs your workbook (`MY_WORKBOOK`) or the packaged scenarios. Before any
   solving, the runner checks the engine's sha256, the runtime, and runs the
   full offline test gate (15-30 min on a 2-CPU Colab). If anything fails it
   refuses to run.
4. Every schedule it publishes has passed the independent validator with 0 hard
   failures. Exit code 0 = approved schedule; anything else = not approved.

**Recommended settings (measured):** `MODE = QUICK`, `SEEDS = 0` (automatic: two
1-hour seeds, best kept). About 1 h on 4+ CPUs, 2 h on 2 CPUs, plus the gate.
`MODE = DEEP` = best of 4 x 1-hour seeds.

**Not recommended:** `SINGLE_LONG_RUN = True`. The one-long-run DEEP path works
(verified end to end) but scored lower than the seeds. **Do not use:** `SKIP_GUARDS = True`.

**Runtime type:** CPU. A GPU does nothing for this solver. Pick High-RAM if offered.

Results are written **straight to Drive**, so a disconnect does not lose a
finished scenario. Put `RC9_2_2_PRODUCTION_PACKAGE.zip` (and your workbook) in
your Drive first.

### If the Drive mount really does hang

The usual causes, in order:

1. **The authorisation popup was blocked.** Allow popups for `colab.research.google.com` and re-run the cell.
2. **Third-party cookies are blocked.** Drive auth needs them. Allow them for Google domains, or use a normal (non-incognito) window.
3. **A stale mount.** Runtime → Restart runtime, then re-run. If it persists, change the mount call to `drive.mount("/content/drive", force_remount=True)`.

**If Drive is just being difficult, stop fighting it** — use the no-Drive notebook instead. You upload the 2 MB ZIP into each session and download the results at the end. For a 45-minute run that is perfectly fine; you only lose the protection against a mid-run disconnect.


In [ ]:
#@title 1. Environment — pin the solver version { display-mode: "form" }
# ortools 9.15.6755 is the pinned RC9.2.2 solver runtime (the runner refuses any other).
# A different solver build can change candidate selection, so pin it.
!pip -q install "ortools==9.15.6755" "openpyxl>=3.1" "scipy>=1.11" "ruff==0.15.8" 2>&1 | tail -2

import multiprocessing, platform, sys
print("python :", platform.python_version())
print("cpus   :", multiprocessing.cpu_count())
import ortools; print("ortools:", ortools.__version__)
import scipy; print("scipy  :", scipy.__version__)

In [ ]:
#@title 2. Mount Drive and locate the ZIP { display-mode: "form" }
# ZIP_PATH is looked at FIRST. Only if it is missing do we search, and then only
# a few shallow locations - never the whole Drive.
#
# The previous version of this cell ran rglob() over /content/drive, which walks
# your ENTIRE Drive over a network filesystem. Every directory listing is a
# round trip, so on a Drive with real content the cell simply never returns and
# looks like the mount is hanging. It was not the mount.
ZIP_NAME = "RC9_2_2_PRODUCTION_PACKAGE.zip"  #@param {type:"string"}
ZIP_PATH = ""  #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")          # add force_remount=True if it misbehaves

import pathlib, itertools
MYDRIVE = pathlib.Path("/content/drive/MyDrive")

def find_zip(name: str):
    """Shallow, bounded search. Never walks the whole Drive."""
    seen = []
    # 1. MyDrive root, 2. a few usual folders, 3. one level down - and stop.
    roots = [MYDRIVE, MYDRIVE / "Colab Notebooks", MYDRIVE / "Downloads"]
    for root in roots:
        candidate = root / name
        if candidate.exists():
            return candidate, seen
        seen.append(str(root))
    try:
        for child in itertools.islice(sorted(p for p in MYDRIVE.iterdir() if p.is_dir()), 40):
            candidate = child / name
            if candidate.exists():
                return candidate, seen
            seen.append(str(child))
    except OSError:
        pass
    return None, seen

if ZIP_PATH.strip():
    found = pathlib.Path(ZIP_PATH.strip())
    assert found.exists(), f"ZIP_PATH does not exist: {found}"
else:
    found, looked = find_zip(ZIP_NAME)
    if found is None:
        print("Could not find", ZIP_NAME, "in:")
        for p in looked[:12]:
            print("   ", p)
        print("\nPut the ZIP in the top level of MyDrive, or paste its full path "
              "into ZIP_PATH above and re-run this cell.")
        raise SystemExit("ZIP not found")

ZIP_PATH = str(found)
print("using:", ZIP_PATH)

In [ ]:
#@title 3. Extract, and point results at Drive { display-mode: "form" }
DRIVE_RESULTS = "/content/drive/MyDrive/RC922_RC5/RESULTS"  #@param {type:"string"}

import zipfile, pathlib, os
os.makedirs("/content/rc922", exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("/content/rc922")

cands = list(pathlib.Path("/content/rc922").rglob("SCENARIOS.json"))
assert cands, "SCENARIOS.json not found after extract"
PACKAGE_ROOT = str(cands[0].parent)

# Results go to Drive so a disconnect cannot lose a finished scenario.
RESULTS_ROOT = pathlib.Path(DRIVE_RESULTS)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RESULTS_ROOT =", RESULTS_ROOT, "(on Drive)")

In [ ]:
#@title 4. Verify the package before running anything { display-mode: "form" }
# Shows what the package contains and which engine it will run. The full
# safety gate (engine sha256, runtime, all offline tests) runs automatically at
# the start of the next step and refuses to run if anything fails.
import subprocess, sys, json, pathlib
ROOT = pathlib.Path(PACKAGE_ROOT)

manifest = json.loads((ROOT / "SCENARIOS.json").read_text())
print("package:", manifest["package"])
print("engine :", manifest["engine_release"])
print("sha256 :", manifest["engine_sha256"][:32], "…")
print()
for row in manifest["scenarios"]:
    print(f"  {row['scenario_id']:14} {row['input']}")
print()

proc = subprocess.run([sys.executable, str(ROOT / "runners" / "rc922_runner.py"),
                       "--package-root", str(ROOT), "--gate-only",
                       "--results-root", str(ROOT / "results")],
                      capture_output=True, text=True)
print("(no results yet is expected on a fresh package)")

In [ ]:
#@title 4b. Check your workbook first (seconds) { display-mode: "form" }
WORKBOOK_TO_CHECK = ""   #@param {type:"string"}
# Full path of the .xlsx you are about to schedule. This runs the same input
# checks the production run starts with, in seconds, so a bad cell is found now
# rather than after the 15-30 min safety gate. ACCEPTED = ready to run.
import subprocess, sys
if WORKBOOK_TO_CHECK.strip():
    proc = subprocess.run([sys.executable, str(ROOT / "tools" / "check_input_workbook.py"),
                           WORKBOOK_TO_CHECK.strip()], capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode:
        print(proc.stderr[-2000:])
        print("Fix the items above in the workbook, upload it again, and re-run this cell.")
else:
    print("Set WORKBOOK_TO_CHECK to the path of your workbook.")

In [ ]:
#@title 5. Run { display-mode: "form" }
MY_WORKBOOK = ""   #@param {type:"string"}
# MY_WORKBOOK: full path of your .xlsx (e.g. /content/drive/MyDrive/schedules/week42.xlsx,
# or /content/week42.xlsx after uploading in the Files panel). Leave empty to run
# the packaged scenarios instead. Results go to RESULTS_ROOT/<WORKBOOK NAME>/.
MODE    = "QUICK"  #@param ["SMOKE", "QUICK", "DEEP", "OVERNIGHT"]
SEEDS   = 0        #@param {type:"integer"}
# SEEDS = 0 is automatic (measured, evidence/seed_portfolio_ab/RESULT.txt):
#   QUICK -> best of 2 x 1 h seeds, DEEP -> best of 4, OVERNIGHT -> best of 6.
# Seeds run side by side on 4+ CPUs and one after another on 2 CPUs.
STAGE   = "FULL_SCHEDULE"  #@param ["FULL_SCHEDULE", "BEFORE_BREAKS_ONLY"]
LANGUAGE_WORKING_WINDOW = "workbook" #@param ["workbook", "OFF", "MINIMUM_ROWS", "ALL_ROWS", "REQUIRED_LANGUAGE_ONLY"]
# "workbook" = use the value in your Instructions sheet (recommended).
# ALL_ROWS keeps every language inside its Language Setup hours (per day with
# Coverage Days); OFF lets anyone work any hour. See PRODUCTION_RUN_GUIDE.md 3.7.
SINGLE_LONG_RUN = False #@param {type:"boolean"}
# Keep False. The one-long-run DEEP/OVERNIGHT path works (verified end to end),
# but best-of-seeds (the default) scored higher in the measurements.
ONLY    = ""       #@param {type:"string"}
SHARD   = 0        #@param {type:"integer"}
SHARDS  = 1        #@param {type:"integer"}
# ONLY / SHARD / SHARDS pick packaged scenarios; ignored when MY_WORKBOOK is set.
TIME_LIMIT = 0     #@param {type:"integer"}
# 0 = the mode's own budget (QUICK 3600 s per seed). MODE sets the budget AND the
# phase reserves, so change MODE rather than TIME_LIMIT.
RESUME = True       #@param {type:"boolean"}
OVERWRITE = False   #@param {type:"boolean"}
# RESUME continues an interrupted run from its checkpoints. OVERWRITE must be
# ticked to replace a workbook/scenario that already has results in RESULTS_ROOT.
SKIP_GUARDS = False #@param {type:"boolean"}
# Keep False: the guards are what make the output safe to use.

import subprocess, sys, multiprocessing, os, signal
cmd = [sys.executable, "-u", str(ROOT / "runners" / "rc922_runner.py"),
       "--package-root", str(ROOT),
       "--results-root", str(RESULTS_ROOT),
       "--shard", str(SHARD), "--shards", str(SHARDS),
       "--mode", MODE,
       "--stage", STAGE,
       "--num-workers", str(multiprocessing.cpu_count())]
AUTO_SEEDS = {"SMOKE": 1, "QUICK": 2, "DEEP": 0, "OVERNIGHT": 0}   # 0: runner default (DEEP 4, OVERNIGHT 6)
seeds = int(SEEDS) if int(SEEDS) > 0 else AUTO_SEEDS[MODE]
if seeds:                            cmd += ["--seeds", str(seeds)]
if SINGLE_LONG_RUN:                  cmd += ["--single-run"]
if TIME_LIMIT:                       cmd += ["--time-limit", str(TIME_LIMIT)]
if ONLY.strip():                     cmd += ["--only", ONLY.strip()]
if MY_WORKBOOK.strip():              cmd += ["--input", MY_WORKBOOK.strip()]
if LANGUAGE_WORKING_WINDOW != "workbook":
    cmd += ["--language-working-window", LANGUAGE_WORKING_WINDOW]
if SKIP_GUARDS:                      cmd += ["--skip-guards"]
if RESUME:                           cmd += ["--resume"]
if OVERWRITE:                        cmd += ["--overwrite"]

print(" ".join(cmd), "\n")
# The runner and everything it starts (seed runs, engines) live in their own
# process group, so interrupting this cell stops all of them, not just the top one.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, start_new_session=(os.name == "posix"))
try:
    for line in proc.stdout:
        print(line, end="")
    print("\nexit code:", proc.wait())
except KeyboardInterrupt:
    print("\nInterrupt received: asking the runner to stop safely and preserve checkpoints...")
    try:
        proc.send_signal(signal.SIGINT)
        print("runner exit code:", proc.wait(timeout=30))
    except subprocess.TimeoutExpired:
        if os.name == "posix":
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        else:
            proc.terminate()
        proc.wait(timeout=30)
    raise


In [ ]:
#@title 6. Score the gates (re-runnable on its own) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, str(ROOT / "tools" / "release_gate_report.py"),
                       str(RESULTS_ROOT), "--out-dir", str(RESULTS_ROOT / "_gate_report")],
                      capture_output=True, text=True)
print(proc.stdout)
if proc.returncode: print(proc.stderr[-3000:])

## 7. Results are already on Drive

Nothing to download — the `DRIVE_RESULTS` folder in your Drive holds every
scenario directory plus `_gate_report/`. Share that folder, or zip it with the cell below
if you prefer a single file.

Writing to Drive is slower than local disk. If a scenario feels slow to publish
its artifacts, that is Drive I/O, not the solver.

In [ ]:
#@title 7. (optional) Zip the Drive results { display-mode: "form" }
import shutil, time, os
stamp = time.strftime("%Y%m%d_%H%M%S")
out = f"/content/drive/MyDrive/RC922_RC5/RESULTS_{stamp}"
shutil.make_archive(out, "zip", str(RESULTS_ROOT))
print("written:", out + ".zip", round(os.path.getsize(out + ".zip")/1e6, 1), "MB")